# Proyecto Final de Inferencia Estadística
## Distance Sampling para estimación de densidad y abundancia

**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958  
**Curso:** Probabilidades e Inferencia Estadística  
**Docente:** Dr. Erick A. Chacón Montalván

Este cuaderno desarrolla el proyecto íntegramente en **Python** y está preparado para ejecutarse en **Google Colab**.

El objetivo es estimar la densidad y abundancia de una población de *Winter Wren* a partir de dos diseños de muestreo por distancias: **point transect** y **line transect**.


## 1. Planteamiento del problema

En un muestreo de fauna, el número de individuos observados suele ser menor que el número realmente presente. La causa es que la probabilidad de detección disminuye con la distancia al observador.

Se define:

$$
D=	ext{distancia del individuo al observador}
$$

y

$$
Z=
\begin{cases}
1,&\text{si el individuo es detectado},\\
0,&\text{si el individuo no es detectado}.
\end{cases}
$$

La meta es reconstruir el mecanismo de detección a partir de las distancias observadas y utilizarlo para corregir los conteos.


## 2. Espacio probabilístico

Para un truncamiento máximo $w$, el experimento elemental se representa mediante

$$
\Omega=[0,w]\times\{0,1\},
$$

$$
\mathcal F=\mathcal B([0,w])\otimes 2^{\{0,1\}},
$$

y una familia paramétrica de medidas de probabilidad

$$
\mathcal P=\{P_\sigma:\sigma>0\}.
$$

La función de detección se modela mediante una half-normal:

$$
g(d;\sigma)
=
P_\sigma(Z=1\mid D=d)
=
\exp\left(-\frac{d^2}{2\sigma^2}\right).
$$

Por tanto,

$$
Z\mid D=d
\sim
\operatorname{Bernoulli}(g(d;\sigma)).
$$

Como las distancias solo se registran cuando $Z=1$, la distribución observada es

$$
D\mid Z=1.
$$


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.special import erf


## 3. Carga de los datos

Los dos archivos se encuentran en el repositorio público del proyecto.

- `wren_5min.csv`: point transect.
- `wren_lt.csv`: line transect.

El cuaderno los descarga directamente desde GitHub para que pueda ejecutarse en Colab sin subir archivos manualmente.


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
url_punto = "https://raw.githubusercontent.com/jantoniootoya/Probabilidades1/main/wren_5min.csv"
url_linea = "https://raw.githubusercontent.com/jantoniootoya/Probabilidades1/main/wren_lt.csv"

punto = pd.read_csv(url_punto)
linea = pd.read_csv(url_linea)

punto.shape, linea.shape


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
display(punto.head())
display(linea.head())


## 4. Preparación del esfuerzo y de las variables

Para point transect:

$$
S_P=K\pi w_P^2,
$$

donde $K$ es el esfuerzo acumulado en visitas a puntos.

Para line transect:

$$
S_L=2w_LL,
$$

donde $L$ es la longitud total recorrida.

Las áreas se convierten a hectáreas para mantener consistencia con el área de estudio.


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
dP = punto["distance"].to_numpy(dtype=float)
dL = linea["distance"].to_numpy(dtype=float)

nP = len(dP)
nL = len(dL)

wP = dP.max()
wL = dL.max()

K = punto[["Sample.Label", "Effort"]].drop_duplicates()["Effort"].sum()
L = linea[["Sample.Label", "Effort"]].drop_duplicates()["Effort"].sum()

area_estudio = float(punto["Area"].iloc[0])

S_P = K * np.pi * wP**2 / 10000
S_L = 2 * wL * (1000 * L) / 10000


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
resumen = pd.DataFrame({
    "Diseño": ["Point transect", "Line transect"],
    "Detecciones": [nP, nL],
    "Unidades": [punto["Sample.Label"].nunique(), linea["Sample.Label"].nunique()],
    "Esfuerzo": [K, L],
    "Distancia mínima": [dP.min(), dL.min()],
    "Distancia máxima": [dP.max(), dL.max()],
    "Media": [dP.mean(), dL.mean()],
    "Mediana": [np.median(dP), np.median(dL)],
    "Área cubierta (ha)": [S_P, S_L]
})

resumen


## 5. Análisis exploratorio

La forma del histograma no debe interpretarse únicamente como un efecto de detectabilidad. La geometría es diferente en cada diseño.

En line transect, la cantidad de área disponible es aproximadamente constante para cada franja de distancia perpendicular.

En point transect, el área disponible aumenta con el radio.


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dP, bins=12)
plt.xlabel("Distancia radial (m)")
plt.ylabel("Frecuencia")
plt.title("Point transect")
plt.show()


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dL, bins=12)
plt.xlabel("Distancia perpendicular (m)")
plt.ylabel("Frecuencia")
plt.title("Line transect")
plt.show()


## 6. Distribución geométrica de la distancia

### Line transect

$$
P(D\leq d)=\frac{d}{w},
$$

por lo que

$$
\pi_L(d)=\frac{1}{w}.
$$

### Point transect

$$
P(D\leq d)=\frac{d^2}{w^2},
$$

por lo que

$$
\pi_P(d)=\frac{2d}{w^2}.
$$


## 7. Probabilidad promedio de detección

La probabilidad promedio de detección es

$$
p(\sigma)
=
P(Z=1)
=
\int_0^w \pi(d)g(d;\sigma)\,dd.
$$

Para line transect:

$$
p_L(\sigma)
=
\frac{\sigma}{w}
\sqrt{\frac{\pi}{2}}
\operatorname{erf}
\left(
\frac{w}{\sqrt{2}\sigma}
\right).
$$

Para point transect:

$$
p_P(\sigma)
=
\frac{2\sigma^2}{w^2}
\left[
1-
\exp\left(-\frac{w^2}{2\sigma^2}\right)
\right].
$$


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def g(d, sigma):
    return np.exp(-(d**2) / (2 * sigma**2))

def pP(sigma):
    return (2 * sigma**2 / wP**2) * (1 - np.exp(-(wP**2) / (2 * sigma**2)))

def pL(sigma):
    return (sigma / wL) * np.sqrt(np.pi / 2) * erf(wL / (np.sqrt(2) * sigma))


## 8. Distribución de las distancias detectadas

Como solo observamos individuos con $Z=1$:

$$
f(d\mid Z=1;\sigma)
=
\frac{\pi(d)g(d;\sigma)}
{p(\sigma)}.
$$

Para line transect:

$$
f_L(d\mid Z=1;\sigma)
=
\frac{
\exp\left(-d^2/(2\sigma^2)\right)
}{
\sigma\sqrt{\pi/2}\,
\operatorname{erf}
\left(w/(\sqrt{2}\sigma)\right)
}.
$$

Para point transect:

$$
f_P(d\mid Z=1;\sigma)
=
\frac{
d\exp\left(-d^2/(2\sigma^2)\right)
}{
\sigma^2
\left[
1-\exp\left(-w^2/(2\sigma^2)\right)
\right]
}.
$$


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def fP(d, sigma):
    return d * np.exp(-(d**2) / (2 * sigma**2)) / (
        sigma**2 * (1 - np.exp(-(wP**2) / (2 * sigma**2)))
    )

def fL(d, sigma):
    return np.exp(-(d**2) / (2 * sigma**2)) / (
        sigma * np.sqrt(np.pi / 2) * erf(wL / (np.sqrt(2) * sigma))
    )


## 9. Máxima verosimilitud

Para las distancias observadas

$$
d_1,\ldots,d_n,
$$

la verosimilitud condicional es

$$
L(\sigma)
=
\prod_{i=1}^{n}
f(d_i\mid Z=1;\sigma).
$$

Los estimadores se obtienen mediante

$$
\widehat\sigma_P
=
\arg\max_{\sigma>0}\ell_P(\sigma),
$$

y

$$
\widehat\sigma_L
=
\arg\max_{\sigma>0}\ell_L(\sigma).
$$


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def loglikP(sigma):
    return (
        np.log(dP).sum()
        - 2 * nP * np.log(sigma)
        - (dP**2).sum() / (2 * sigma**2)
        - nP * np.log(1 - np.exp(-(wP**2) / (2 * sigma**2)))
    )

def loglikL(sigma):
    return (
        -(dL**2).sum() / (2 * sigma**2)
        - nL * np.log(
            sigma * np.sqrt(np.pi / 2) * erf(wL / (np.sqrt(2) * sigma))
        )
    )


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
ajuste_P = minimize_scalar(lambda sigma: -loglikP(sigma), bounds=(0.1, 500), method="bounded")
ajuste_L = minimize_scalar(lambda sigma: -loglikL(sigma), bounds=(0.1, 500), method="bounded")

sigma_P = ajuste_P.x
sigma_L = ajuste_L.x

sigma_P, sigma_L


## 10. Probabilidad de detección, densidad y abundancia

Una vez estimado $\sigma$:

$$
\widehat p=p(\widehat\sigma).
$$

La densidad se estima mediante

$$
\widehat A
=
\frac{n}{S\widehat p},
$$

y la abundancia total mediante

$$
\widehat N
=
\widehat A A_{\mathrm{study}}.
$$


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
p_det_P = pP(sigma_P)
p_det_L = pL(sigma_L)

A_P = nP / (S_P * p_det_P)
A_L = nL / (S_L * p_det_L)

N_P = A_P * area_estudio
N_L = A_L * area_estudio

estimaciones = pd.DataFrame({
    "Diseño": ["Point transect", "Line transect"],
    "sigma": [sigma_P, sigma_L],
    "Probabilidad de detección": [p_det_P, p_det_L],
    "Densidad por ha": [A_P, A_L],
    "Abundancia": [N_P, N_L]
})

estimaciones


## 11. Curvas de detección estimadas

La curva permite interpretar directamente el parámetro $\widehat\sigma$. Un valor mayor implica una caída más lenta de la probabilidad de detección con la distancia.


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
xP = np.linspace(0, wP, 400)

plt.figure(figsize=(8, 5))
plt.plot(xP, g(xP, sigma_P))
plt.xlabel("Distancia (m)")
plt.ylabel("Probabilidad de detección")
plt.ylim(0, 1)
plt.title("Función de detección estimada — Point transect")
plt.show()


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
xL = np.linspace(0, wL, 400)

plt.figure(figsize=(8, 5))
plt.plot(xL, g(xL, sigma_L))
plt.xlabel("Distancia (m)")
plt.ylabel("Probabilidad de detección")
plt.ylim(0, 1)
plt.title("Función de detección estimada — Line transect")
plt.show()


## 12. Verificación visual del modelo de distancias observadas

La densidad estimada de $D\mid Z=1$ se compara con el histograma normalizado de las distancias registradas.


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dP, bins=12, density=True, alpha=0.5)
plt.plot(xP[1:], fP(xP[1:], sigma_P))
plt.xlabel("Distancia radial (m)")
plt.ylabel("Densidad")
plt.title("Ajuste de distancias — Point transect")
plt.show()


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dL, bins=12, density=True, alpha=0.5)
plt.plot(xL, fL(xL, sigma_L))
plt.xlabel("Distancia perpendicular (m)")
plt.ylabel("Densidad")
plt.title("Ajuste de distancias — Line transect")
plt.show()


## 13. Inferencia conjunta para densidad y detección

Para cuantificar la incertidumbre de $A$ se introduce explícitamente un supuesto adicional: un proceso espacial homogéneo de Poisson.

Si $S$ es el área cubierta:

$$
N(S)\sim\operatorname{Poisson}(AS).
$$

Después del mecanismo de detección:

$$
n\sim\operatorname{Poisson}(ASp(\sigma)).
$$

La log-verosimilitud conjunta, omitiendo constantes que no dependen de los parámetros, es

$$
\ell(A,\sigma)
=
n\log(ASp(\sigma))
-
ASp(\sigma)
+
\sum_{i=1}^{n}
\log f(d_i\mid Z=1;\sigma).
$$

Se trabaja con

$$
\eta_A=\log A,
\qquad
\eta_\sigma=\log\sigma,
$$

para garantizar positividad.

La incertidumbre se obtiene mediante la información observada:

$$
J(\widehat\eta)
=
-\nabla^2\ell(\widehat\eta),
$$

$$
\widehat{\operatorname{Var}}(\widehat\eta)
=
J(\widehat\eta)^{-1}.
$$

En este cuaderno no se utiliza bootstrap.


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def loglik_conjunta_P(theta):
    A, sigma = np.exp(theta)
    p = pP(sigma)
    return nP * np.log(A * S_P * p) - A * S_P * p + loglikP(sigma)

def loglik_conjunta_L(theta):
    A, sigma = np.exp(theta)
    p = pL(sigma)
    return nL * np.log(A * S_L * p) - A * S_L * p + loglikL(sigma)


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
def hessiano(f, x, h=1e-4):
    x = np.asarray(x, dtype=float)
    m = len(x)
    H = np.zeros((m, m))
    fx = f(x)

    for i in range(m):
        ei = np.zeros(m)
        ei[i] = h
        H[i, i] = (f(x + ei) - 2 * fx + f(x - ei)) / h**2

        for j in range(i + 1, m):
            ej = np.zeros(m)
            ej[j] = h
            H[i, j] = (
                f(x + ei + ej)
                - f(x + ei - ej)
                - f(x - ei + ej)
                + f(x - ei - ej)
            ) / (4 * h**2)
            H[j, i] = H[i, j]

    return H


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
theta_P = np.log([A_P, sigma_P])
theta_L = np.log([A_L, sigma_L])

V_P = np.linalg.inv(-hessiano(loglik_conjunta_P, theta_P))
V_L = np.linalg.inv(-hessiano(loglik_conjunta_L, theta_L))

se_P = np.sqrt(np.diag(V_P))
se_L = np.sqrt(np.diag(V_L))

IC_A_P = np.exp([theta_P[0] - 1.96 * se_P[0], theta_P[0] + 1.96 * se_P[0]])
IC_A_L = np.exp([theta_L[0] - 1.96 * se_L[0], theta_L[0] + 1.96 * se_L[0]])

IC_sigma_P = np.exp([theta_P[1] - 1.96 * se_P[1], theta_P[1] + 1.96 * se_P[1]])
IC_sigma_L = np.exp([theta_L[1] - 1.96 * se_L[1], theta_L[1] + 1.96 * se_L[1]])

IC_p_P = pP(IC_sigma_P)
IC_p_L = pL(IC_sigma_L)

IC_N_P = IC_A_P * area_estudio
IC_N_L = IC_A_L * area_estudio


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
resultados = pd.DataFrame({
    "Diseño": ["Point transect", "Line transect"],
    "n": [nP, nL],
    "w (m)": [wP, wL],
    "sigma": [sigma_P, sigma_L],
    "IC95 sigma LI": [IC_sigma_P[0], IC_sigma_L[0]],
    "IC95 sigma LS": [IC_sigma_P[1], IC_sigma_L[1]],
    "p": [p_det_P, p_det_L],
    "IC95 p LI": [IC_p_P[0], IC_p_L[0]],
    "IC95 p LS": [IC_p_P[1], IC_p_L[1]],
    "Densidad": [A_P, A_L],
    "IC95 A LI": [IC_A_P[0], IC_A_L[0]],
    "IC95 A LS": [IC_A_P[1], IC_A_L[1]],
    "Abundancia": [N_P, N_L],
    "IC95 N LI": [IC_N_P[0], IC_N_L[0]],
    "IC95 N LS": [IC_N_P[1], IC_N_L[1]]
})

resultados.round(4)


## 14. Comparación entre diseños

El objetivo final no es esperar que ambos diseños produzcan exactamente el mismo $\widehat\sigma$, porque la geometría del muestreo es diferente. La comparación relevante se centra en las estimaciones de densidad y abundancia de la misma población.


**Autor:** José Antonio Otoya Barrenechea  
**Programa:** Maestría en Ciencias e Ingeniería Estadística — Universidad Nacional de Ingeniería  
**ORCID:** 0009-0007-0702-6958


In [ ]:
LI_comun = max(IC_A_P[0], IC_A_L[0])
LS_comun = min(IC_A_P[1], IC_A_L[1])

if LI_comun <= LS_comun:
    comparacion = "Los intervalos de confianza de densidad se superponen."
else:
    comparacion = "Los intervalos de confianza de densidad no se superponen."

comparacion


## 15. Resultados esperados como control de ejecución

Si el cuaderno se ejecuta correctamente con los archivos actuales, los estimadores deben quedar aproximadamente alrededor de:

- Point transect: $\widehat\sigma\approx 43.37$.
- Line transect: $\widehat\sigma\approx 60.69$.
- Line transect: $\widehat p\approx 0.685$.

Estos valores sirven únicamente como control reproducible de que el código se ejecutó correctamente.


## 16. Supuestos y limitaciones

1. La densidad espacial se considera homogénea dentro del área analizada.
2. Se asume detección perfecta a distancia cero: $g(0)=1$.
3. La detectabilidad se modela mediante una función half-normal.
4. Se supone que las distancias están correctamente registradas.
5. Se utiliza la máxima distancia registrada como $w$ en cada base.
6. Para cuantificar conjuntamente la incertidumbre de densidad y detección se adopta un proceso espacial homogéneo de Poisson.
7. Las distancias de point transect muestran discretización o redondeo, mientras el modelo se desarrolla como continuo siguiendo el planteamiento matemático del proyecto.


## 17. Conclusión metodológica

La secuencia inferencial completa es:

$$
(\Omega,\mathcal F,\mathcal P)
\longrightarrow
D
\longrightarrow
Z\mid D
\longrightarrow
D\mid Z=1
\longrightarrow
L(A,\sigma)
\longrightarrow
(\widehat A,\widehat\sigma)
\longrightarrow
\widehat N.
$$

La metodología separa el proceso espacial de los individuos del mecanismo imperfecto de observación. La función de detección permite estimar qué proporción de la población es observada y corregir el conteo para obtener densidad y abundancia.


## Referencias

Buckland, S. T. (2006). Point-Transect Surveys for Songbirds: Robust Methodologies. *The Auk, 123*(2), 345–357.

Buckland, S. T., Anderson, D. R., Burnham, K. P., Laake, J. L., Borchers, D. L., & Thomas, L. (2001). *Introduction to Distance Sampling: Estimating Abundance of Biological Populations*. Oxford University Press.

Thomas, L., Buckland, S. T., Rexstad, E. A., et al. (2010). Distance software: design and analysis of distance sampling surveys for estimating population size. *Journal of Applied Ecology, 47*(1), 5–14.
